# Reward model — scatter analysis (true vs predicted)

Analisi di un **checkpoint del reward model** addestrato col `demo_algorithm`. Per ogni sorgente di dati produciamo scatter plot con:

- **asse x = reward PREDETTA** dal reward model caricato dal checkpoint;
- **asse y = reward VERA** (`true_reward`).

> Nota sugli assi: questa convenzione (x=pred, y=true) è quella richiesta ed è **invertita** rispetto ai notebook `plot_test*` esistenti.

## Sorgenti analizzate
1. **Traiettorie esperto** — `data_for_training/expert_trajectories.pkl` (`list[Trajectory]`).
2. **Rollout corrente dell'agente** — campionato dall'`agent.zip` del checkpoint via `TrajectoryGeneratorFromAgent`.
3. **Debug dataset, livello traiettoria** — `data_for_training/debug_dataset_full_ep.pkl` (episodi completi, prodotto da `extract_debug_episodes.ipynb`).
4. **Debug dataset, livello transizione** — `data_for_training/debug_dataset.pkl` (lista piatta di `Transition`).

## Normalizzazione (gestione lunghezze diverse / scale diverse)
Reward vera e predetta vivono su scale diverse (il reward model è raw, non normalizzato): la retta `y=x`, RMSE e MAE hanno senso solo dopo aver portato la predetta sulla scala della vera. Pearson/Spearman/Kendall sono invece scale-invariant.

**Allineamento assi (sempre, come `plot_test`)** — la predetta viene allineata alla vera:
- `matching_mean=True` → mean-shift: `pred - mean(pred) + mean(true)`;
- `matching_mean=True, matching_std=True` → z-score completo: `(pred-mean(pred))/std(pred) * std(true) + mean(true)`.

**Livello traiettoria — due opzioni di normalizzazione del ritorno** (toggle indipendenti, episodi di lunghezza diversa):
- `divide_by_length=True` → *divisione per lunghezza dopo la somma*: ritorno medio per step `R/len`;
- z-score *per ritorno di traiettoria* → ottenuta con `matching_mean=True, matching_std=True` applicati ai ritorni (la predetta viene z-scorata sulla scala dei ritorni veri).

Le statistiche di allineamento sono calcolate **sul sottoinsieme effettivamente plottato** (es. solo transizioni `running`), così la nuvola si centra su `y=x`.

> **Ambiente:** questo notebook richiede l'ambiente con `human_feedback_rl`, `stable_baselines3`, `sumo_rl_ego`/`traci` (il rollout dell'agente apre l'env SUMO). Va eseguito sulla macchina di training; non è eseguibile su un Mac senza tale ambiente.

In [ ]:
from pathlib import Path
import pickle

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr, kendalltau, weightedtau
from stable_baselines3 import PPO, SAC

import sumo_rl_ego as sre
from human_feedback_rl.common.types import Trajectory, Transition
from human_feedback_rl.common.trajectory_generators import TrajectoryGeneratorFromAgent

from loadings import load_reward_ensemble

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120

In [ ]:
# ============================ CONFIGURAZIONE ============================
# Cartella del run e nome del checkpoint da analizzare. Modifica liberamente.
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RUN_DIR = REPO / "outputs" / "demo_sac" / "sac_demo_irl loss=maxent_corrected relabel=False seed=0"
CHECKPOINT_NAME = "checkpoint_0060"          # es. checkpoint_0010 ... checkpoint_0060
CKPT_DIR = RUN_DIR / CHECKPOINT_NAME

DATA_DIR = REPO / "data_for_training"
EXPERT_PKL        = DATA_DIR / "expert_trajectories.pkl"
DEBUG_FULL_EP_PKL = DATA_DIR / "debug_dataset_full_ep.pkl"   # prodotto da extract_debug_episodes.ipynb
DEBUG_TRANS_PKL   = DATA_DIR / "debug_dataset.pkl"           # lista piatta di Transition

N_ROLLOUT_STEPS = 100_000   # n. di transizioni da campionare per il rollout dell'agente

for p in (CKPT_DIR / "reward_model.pt", CKPT_DIR / "agent.zip", EXPERT_PKL):
    assert p.exists(), f"Manca: {p}"
print("Checkpoint:", CKPT_DIR)

In [ ]:
# ============================ ENV SUMO ============================
# (Replica del setup dei notebook esistenti: monkeypatch di load_traci e vec env.)
import sumo_gym_ego.core.simulation as _sim_mod
import traci as _traci_mod
_sim_mod.load_traci = lambda use_gui: _traci_mod

try:
    env.close()
except Exception:
    pass

env = sre.make_vec_env(
    "HighwayEgo-v0",
    n_envs=4,
    base_seed=0,
    ego="continuous",
    reward="fast",
)
print("obs_space:", env.observation_space, "| act_space:", env.action_space)

In [ ]:
# ============================ CARICAMENTO CHECKPOINT ============================
reward_model = load_reward_ensemble(
    CKPT_DIR / "reward_model.pt",
    env.observation_space,
    env.action_space,
    device="cpu",
)


def load_agent(path, env, device="cpu"):
    """Carica l'agente rilevando l'algoritmo (demo_sac -> SAC, ppo_demo_irl -> PPO)."""
    last_err = None
    for Algo in (SAC, PPO):
        try:
            agent = Algo.load(path, env=env, device=device)
            print(f"Agente caricato come {Algo.__name__}")
            return agent
        except Exception as e:  # algoritmo non corrispondente: provo il successivo
            last_err = e
    raise last_err


agent = load_agent(CKPT_DIR / "agent.zip", env=env, device="cpu")
print("Reward model e agente caricati da", CHECKPOINT_NAME)

In [ ]:
# ============================ HELPER ============================
STATUS_ARRIVED, STATUS_COLLIDED, STATUS_OFFROAD, STATUS_TIMEOUT, STATUS_RUNNING = 0, 1, 2, 3, 4
STATUS_LABELS = {0: "arrived", 1: "collision", 2: "off_road", 3: "timeout",
                 4: "running", 5: "teleported", 6: "removed"}
STATUS_COLORS = {0: "steelblue", 1: "orange", 2: "crimson", 3: "mediumpurple",
                 4: "gray", 5: "gold", 6: "saddlebrown"}


def _flatten(source):
    """Accetta list[Trajectory]/list[list[Transition]] oppure lista piatta di Transition."""
    if len(source) == 0:
        return []
    if isinstance(source[0], Transition):
        return list(source)
    return [t for traj in source for t in traj]


def transitions_to_arrays(transitions):
    obs    = np.array([t.observation for t in transitions], dtype=np.float32)
    acts   = np.array([t.action      for t in transitions], dtype=np.float32)
    status = np.array([t.next_status for t in transitions], dtype=np.float32)
    done   = np.array([float(t.done) for t in transitions], dtype=np.float32)
    true_r = np.array([t.true_reward for t in transitions], dtype=np.float32)
    return obs, acts, status, done, true_r


def _zscore(x):
    """Standardizza un vettore con la PROPRIA media/std (std protetta da 0)."""
    x = np.asarray(x, float)
    s = x.std()
    return (x - x.mean()) / (s if s > 1e-12 else 1.0)


def align_pred_to_true(true, pred, matching_mean, matching_std):
    """Allinea la predetta alla scala della vera (affine match cross-asse)."""
    true, pred = np.asarray(true, float), np.asarray(pred, float)
    if matching_mean and matching_std:
        ps = pred.std()
        ps = ps if ps > 1e-12 else 1.0
        return (pred - pred.mean()) / ps * true.std() + true.mean()
    if matching_mean:
        return pred - pred.mean() + true.mean()
    return pred


def _ccc(x, y):
    """Lin's Concordance Correlation Coefficient."""
    mx, my = np.mean(x), np.mean(y)
    vx, vy = np.var(x), np.var(y)
    cov = np.cov(x, y, ddof=0)[0, 1]
    return (2 * cov) / (vx + vy + (mx - my) ** 2)


def _metrics_text(true, pred, n):
    pr, _ = pearsonr(true, pred)
    sr, _ = spearmanr(true, pred)
    kt, _ = kendalltau(true, pred)
    wt, _ = weightedtau(true, pred)
    rmse = np.sqrt(np.mean((pred - true) ** 2))
    mae  = np.mean(np.abs(pred - true))
    line1 = f"Pearson r={pr:.3f}   Spearman ρ={sr:.3f}   Kendall τ={kt:.3f}   n={n}"
    line2 = f"CCC={_ccc(true, pred):.3f}   RMSE={rmse:.4f}   MAE={mae:.4f}   Weighted-τ={wt:.3f}"
    return f"{line1}\n{line2}"


def _scatter(pred, true, status_id_per_point, title, norm_tag):
    """Disegno comune: x=predetta, y=vera, colore per status, retta y=x."""
    fig, ax = plt.subplots(figsize=(7, 6))
    for sid, color in STATUS_COLORS.items():
        m = status_id_per_point == sid
        if not m.any():
            continue
        ax.scatter(pred[m], true[m], color=color, label=STATUS_LABELS[sid],
                   alpha=0.7, s=20, edgecolors="none")
    lo = float(min(pred.min(), true.min()))
    hi = float(max(pred.max(), true.max()))
    ax.plot([lo, hi], [lo, hi], "r--", lw=1, label="y = x")
    ax.set_xlabel(f"Predicted reward ({norm_tag})")
    ax.set_ylabel("True reward")
    ax.set_title(f"{title}\n{_metrics_text(true, pred, len(true))}", fontsize=8.5)
    ax.legend(fontsize=8, markerscale=1.2)
    plt.tight_layout()
    plt.show()
    return fig, ax


def scatter_transition_level(source, reward_model, *, matching_mean=True, matching_std=False,
                             status_filter=(STATUS_RUNNING,), title=""):
    """Uno scatter point per ogni TRANSIZIONE.

    status_filter: tuple di id status da plottare (default solo 'running' per evitare
    che le transizioni terminali, dominate da grandi bonus/penalità, distorcano la nuvola).
    Passa None per plottare tutte le transizioni.
    """
    transitions = _flatten(source)
    obs, acts, status, done, true_r = transitions_to_arrays(transitions)
    pred_r = reward_model.predict(obs, acts, status, done)

    terminal_status = np.argmax(status, axis=1)
    mask = np.ones(len(transitions), bool) if status_filter is None \
        else np.isin(terminal_status, status_filter)

    true_p = true_r[mask]
    pred_p = align_pred_to_true(true_p, pred_r[mask], matching_mean, matching_std)
    norm_tag = ("z-score→true" if (matching_mean and matching_std)
                else "mean-shift→true" if matching_mean else "raw")
    return _scatter(pred_p, true_p, terminal_status[mask], f"[transition] {title}", norm_tag)


def trajectory_returns(trajectories, reward_model):
    """Ritorni per-episodio (somma reward), lunghezze e status terminale."""
    true_ret, pred_ret, lengths, term = [], [], [], []
    for ep in trajectories:
        obs, acts, status, done, true_r = transitions_to_arrays(ep)
        pred_r = reward_model.predict(obs, acts, status, done)
        true_ret.append(float(true_r.sum()))
        pred_ret.append(float(pred_r.sum()))
        lengths.append(len(ep))
        term.append(int(np.argmax(ep[-1].next_status)))
    return (np.array(true_ret), np.array(pred_ret),
            np.array(lengths, float), np.array(term))


def scatter_trajectory_level(trajectories, reward_model, *, divide_by_length=True,
                             standardize="zscore_each", title=""):
    """Uno scatter point per ogni TRAIETTORIA (episodio).

    Due normalizzazioni INDIPENDENTI e componibili:

    divide_by_length : bool
        Se True, ritorno -> somma/lunghezza (reward MEDIO per step). Toglie l'effetto
        della lunghezza. Applicato identicamente a True e Pred (trasformazione per-punto).

    standardize : str
        "none"          -> nessuna standardizzazione/allineamento (assi su scale diverse).
        "align_to_true" -> la PREDETTA viene z-scorata sulla scala della VERA (cross-asse);
                           la vera resta nelle sue unita'.
        "zscore_each"   -> 'z-score per ritorno': ENTRAMBI gli assi standardizzati con la
                           PROPRIA media/std. E' componibile con divide_by_length
                           (prima /len, poi z-score). Pearson/Spearman invariati; cambia
                           solo la resa visiva (nuvola centrata in 0, std 1 su ciascun asse).
    """
    true_ret, pred_ret, lengths, term = trajectory_returns(trajectories, reward_model)

    if divide_by_length:
        true_ret = true_ret / lengths
        pred_ret = pred_ret / lengths

    if standardize == "zscore_each":
        true_plot, pred_plot = _zscore(true_ret), _zscore(pred_ret)
        std_tag = "z-score/asse"
    elif standardize == "align_to_true":
        true_plot = true_ret
        pred_plot = align_pred_to_true(true_ret, pred_ret, matching_mean=True, matching_std=True)
        std_tag = "z-score→true"
    elif standardize == "none":
        true_plot, pred_plot = true_ret, pred_ret
        std_tag = "raw"
    else:
        raise ValueError(f"standardize sconosciuto: {standardize!r}")

    len_tag = "mean/step" if divide_by_length else "sum"
    return _scatter(pred_plot, true_plot, term, f"[trajectory|{len_tag}] {title}",
                    f"{len_tag}, {std_tag}")

In [ ]:
# ===== Estensioni per la sezione Debug a livello traiettoria =====
# 1) scatter a livello traiettoria con l'ID dell'episodio scritto DENTRO ogni punto;
# 2) griglia delle curve reward-per-step (true vs predicted) con affine matching
#    PER TRAIETTORIA. Gli ID coincidono tra scatter e curve.


def scatter_trajectory_level_ids(trajectories, reward_model, ids, *,
                                 divide_by_length=True, standardize="zscore_each", title=""):
    """Come scatter_trajectory_level, ma annota ogni punto con l'id dell'episodio."""
    true_ret, pred_ret, lengths, term = trajectory_returns(trajectories, reward_model)
    if divide_by_length:
        true_ret = true_ret / lengths
        pred_ret = pred_ret / lengths

    if standardize == "zscore_each":
        true_plot, pred_plot = _zscore(true_ret), _zscore(pred_ret)
        std_tag = "z-score/asse"
    elif standardize == "align_to_true":
        true_plot = true_ret
        pred_plot = align_pred_to_true(true_ret, pred_ret, matching_mean=True, matching_std=True)
        std_tag = "z-score→true"
    elif standardize == "none":
        true_plot, pred_plot = true_ret, pred_ret
        std_tag = "raw"
    else:
        raise ValueError(f"standardize sconosciuto: {standardize!r}")
    len_tag = "mean/step" if divide_by_length else "sum"

    fig, ax = plt.subplots(figsize=(8, 7))
    for sid, color in STATUS_COLORS.items():
        m = term == sid
        if not m.any():
            continue
        ax.scatter(pred_plot[m], true_plot[m], color=color, label=STATUS_LABELS[sid],
                   alpha=0.85, s=170, edgecolors="white", linewidths=0.6, zorder=2)
    for x, y, i in zip(pred_plot, true_plot, ids):
        ax.text(x, y, str(i), ha="center", va="center", fontsize=6, color="black", zorder=3)

    lo = float(min(pred_plot.min(), true_plot.min()))
    hi = float(max(pred_plot.max(), true_plot.max()))
    ax.plot([lo, hi], [lo, hi], "r--", lw=1, label="y = x", zorder=1)
    ax.set_xlabel(f"Predicted reward ({len_tag}, {std_tag})")
    ax.set_ylabel("True reward")
    ax.set_title(f"[trajectory|{len_tag}] {title}  (id nei punti)\n"
                 f"{_metrics_text(true_plot, pred_plot, len(true_plot))}", fontsize=8.5)
    ax.legend(fontsize=8, markerscale=0.5)
    plt.tight_layout()
    plt.show()
    return fig, ax


def plot_reward_curves_per_trajectory(trajectories, reward_model, ids=None, *,
                                      ncols=2, show_sigma=True):
    """Per ogni episodio: reward PER STEP, true vs predicted.

    Affine matching PER TRAIETTORIA: la predetta di ogni episodio viene allineata
    (mean/std) alla PROPRIA true; le statistiche di allineamento sono calcolate sulle
    sole transizioni 'running' (così la penalità terminale non gonfia la scala).
    L'id nel titolo coincide con quello dello scatter annotato.
    """
    n = len(trajectories)
    if ids is None:
        ids = list(range(n))
    nrows = -(-n // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 7.0, 2.5 * nrows), squeeze=False)
    for ax in axes.flat:
        ax.set_visible(False)

    for k, (traj, ep_id) in enumerate(zip(trajectories, ids)):
        ax = axes.flat[k]
        ax.set_visible(True)
        obs, acts, status, done, true_r = transitions_to_arrays(traj)
        pred_mean, pred_std = reward_model.predict_mean_std(obs, acts, status, done)

        running = status[:, STATUS_RUNNING] == 1
        ref = running if running.sum() >= 2 else np.ones(len(traj), dtype=bool)
        t_mean, t_std = true_r[ref].mean(), true_r[ref].std()
        p_mean, p_std = pred_mean[ref].mean(), pred_mean[ref].std()
        scale = (t_std if t_std > 1e-12 else 1.0) / (p_std if p_std > 1e-12 else 1.0)
        pred_al   = (pred_mean - p_mean) * scale + t_mean   # affine: mean+std -> true
        pred_s_al = pred_std * scale

        steps = np.arange(len(traj))
        ax.plot(steps, true_r, color="royalblue", lw=1.2, label="True")
        ax.plot(steps, pred_al, color="tomato", lw=1.2, ls="--", label="Pred (affine→true)")
        if show_sigma:
            ax.fill_between(steps, pred_al - pred_s_al, pred_al + pred_s_al,
                            color="tomato", alpha=0.18, label="±1σ ensemble")

        try:
            pr, _ = pearsonr(true_r[ref], pred_mean[ref])
        except Exception:
            pr = np.nan
        term = int(np.argmax(traj[-1].next_status))
        ax.set_title(f"id={ep_id} | {STATUS_LABELS[term]} | L={len(traj)} | r_run={pr:.2f}",
                     fontsize=8)
        ax.tick_params(labelsize=6)
        ax.set_xlabel("Step", fontsize=7)
        ax.set_ylabel("Reward", fontsize=7)
        if k == 0:
            ax.legend(fontsize=7, loc="best")

    plt.suptitle("Debug — reward per step: true vs predicted (affine matching per traiettoria)",
                 y=1.001, fontsize=12)
    plt.tight_layout()
    plt.show()

## 1. Traiettorie esperto
Livello traiettoria (ritorno per episodio) e livello transizione.

In [ ]:
with open(EXPERT_PKL, "rb") as f:
    expert_trajs = pickle.load(f)
print(f"Esperto: {len(expert_trajs)} traiettorie, {sum(len(t) for t in expert_trajs)} transizioni")

# Livello traiettoria — reward medio per step (divide_by_length) + z-score per asse.
scatter_trajectory_level(expert_trajs, reward_model,
                         divide_by_length=True, standardize="zscore_each",
                         title="expert")

# Livello transizione — solo running, allineamento mean-shift
scatter_transition_level(expert_trajs, reward_model,
                         matching_mean=True, matching_std=False,
                         status_filter=(STATUS_RUNNING,), title="expert")

## 2. Rollout corrente dell'agente
Campionato dall'`agent.zip` del checkpoint. Richiede l'env SUMO attivo.

## 3. Debug dataset — livello traiettoria
Usa `debug_dataset_full_ep.pkl` (episodi completi prodotti da `extract_debug_episodes.ipynb`).

Ogni episodio riceve un **id** `0..79`, usato in modo coerente in entrambi gli output:

- **(a) Scatter livello traiettoria** con l'**id scritto dentro ogni punto** (reward medio per step, z-score per asse).
- **(b) Le 80 curve** *reward-per-step* (true vs predicted), con **affine matching per traiettoria**: la predetta di ogni episodio è allineata (mean/std) alla propria true, con statistiche calcolate sulle sole transizioni `running` (la penalità terminale non gonfia la scala). Nel titolo: `id | status terminale | lunghezza | r_run` (Pearson sulle running).

In [ ]:
assert DEBUG_FULL_EP_PKL.exists(), (
    f"{DEBUG_FULL_EP_PKL} non trovato: esegui prima extract_debug_episodes.ipynb")
with open(DEBUG_FULL_EP_PKL, "rb") as f:
    debug_eps = pickle.load(f)
print(f"Debug episodi: {len(debug_eps)} traiettorie, "
      f"{sum(len(t) for t in debug_eps)} transizioni")

# Id 0..79: stesso id usato nello scatter (dentro i punti) e nelle 80 curve.
debug_ids = list(range(len(debug_eps)))

# (a) Scatter livello traiettoria con l'id dell'episodio scritto dentro ogni punto.
scatter_trajectory_level_ids(debug_eps, reward_model, debug_ids,
                             divide_by_length=True, standardize="zscore_each",
                             title="debug full-ep")

# (b) 80 curve: reward per step (true vs predicted), affine matching per traiettoria.
#     Figura molto alta (40 righe x 2 colonne): regola ncols se preferisci.
plot_reward_curves_per_trajectory(debug_eps, reward_model, ids=debug_ids, ncols=2)

## 4. Debug dataset — livello transizione
Usa `debug_dataset.pkl` (lista piatta di `Transition`).

In [ ]:
with open(DEBUG_TRANS_PKL, "rb") as f:
    debug_transitions = pickle.load(f)
assert isinstance(debug_transitions[0], Transition), "Atteso lista piatta di Transition"
print(f"Debug transizioni: {len(debug_transitions)}")

# Solo running (escludendo i terminali dominati da bonus/penalità)
scatter_transition_level(debug_transitions, reward_model,
                         matching_mean=True, matching_std=False,
                         status_filter=(STATUS_RUNNING,), title="debug transitions")

# Variante: tutte le transizioni, colorate per status
scatter_transition_level(debug_transitions, reward_model,
                         matching_mean=True, matching_std=False,
                         status_filter=None, title="debug transitions (all)")